# Experiment 7 — Pulse shaping and the Nyquist criterion

**Arkaprava Dutta · B.Tech ECE · Cooch Behar Government Engineering College**

Runs unchanged in **Google Colab** (`Runtime → Run all`) or in any local Jupyter / Python 3 setup.
Only `numpy` and `matplotlib` are required, both preinstalled in Colab.

**Implementation tasks covered**

| Task | Description | Cell |
|---|---|---|
| 23 | Upsampled symbol impulse train | §3, Figure 3 |
| 24 | Roll-offs α = 0, 0.25, 0.5, 1 | §5, Figures 1–4 |
| 25 | Impulse and frequency responses | Figures 1–2 |
| 26 | Verify zero crossings at symbol intervals | Validation 1 |
| 27 | Cascade Tx and Rx RRC filters | Validation 2, Figure 3 |


## 1. Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

rng = np.random.default_rng(7)     # fixed seed -> reproducible symbol stream

T      = 1.0        # symbol period (s)
sps    = 16         # samples per symbol
fs     = sps / T    # sampling rate
Ts     = 1.0 / fs
SPAN   = 10         # filter span in symbols (+/- SPAN/2)
ALPHAS = [0.0, 0.25, 0.5, 1.0]

SAVE = False        # True -> also write PNGs and the validation text file
OUT  = "."

print(f"fs = {fs} Hz, filter length = {int(SPAN*sps)+1} taps")


## 2. Pulse definitions

Every filter is generated from its closed-form mathematical definition — no library
filter-design call. The 0/0 points are replaced by their analytic limits; leave them
out and those taps become `NaN`, which silently destroys the whole convolution.

| Filter | Singular points | Limit |
|---|---|---|
| RC | t = 0 | 1 |
| RC | t = ±T/2α | (π/4)·sinc(1/2α) |
| RRC | t = 0 | 1 + α(4/π − 1) |
| RRC | t = ±T/4α | (α/√2)[(1+2/π)sin(π/4α) + (1−2/π)cos(π/4α)] |


In [ ]:
def sinc_pulse(t, T=T):
    """Ideal Nyquist pulse  h(t) = sin(pi t/T)/(pi t/T).  Limit at t=0 is 1."""
    x = t / T
    out = np.ones_like(x)
    nz = np.abs(x) > 1e-12                      # removable singularity at t = 0
    out[nz] = np.sin(np.pi * x[nz]) / (np.pi * x[nz])
    return out


def rect_pulse(t, T=T):
    """NRZ rectangular pulse of unit height over |t| < T/2."""
    # half-open support: avoids double-counting the shared edge sample
    return np.where((t >= -T / 2) & (t < T / 2), 1.0, 0.0)


def raised_cosine(t, alpha, T=T):
    """
    Raised-cosine impulse response

        h(t) = sinc(t/T) * cos(pi*alpha*t/T) / (1 - (2*alpha*t/T)^2)

    Removable singularities
        t = 0          ->  h = 1
        t = +-T/(2a)   ->  h = (pi/4) * sinc(1/(2a))      [L'Hopital]
    """
    t = np.asarray(t, dtype=float)
    x = t / T
    h = np.empty_like(x)

    sing0 = np.abs(x) < 1e-12
    if alpha > 0:
        sing1 = np.abs(np.abs(x) - 1.0 / (2 * alpha)) < 1e-9
    else:
        sing1 = np.zeros_like(x, dtype=bool)
    good = ~(sing0 | sing1)

    xg = x[good]
    h[good] = (np.sin(np.pi * xg) / (np.pi * xg)) * \
              np.cos(np.pi * alpha * xg) / (1.0 - (2 * alpha * xg) ** 2)

    h[sing0] = 1.0
    if alpha > 0 and sing1.any():
        h[sing1] = (np.pi / 4.0) * np.sinc(1.0 / (2 * alpha))
    return h


def root_raised_cosine(t, alpha, T=T):
    """
    Root-raised-cosine impulse response (unit-amplitude convention, 1/T scaling
    dropped so that alpha->0 reduces exactly to sinc(t/T)):

        h(t) = [ sin(pi*t/T*(1-a)) + 4a*t/T * cos(pi*t/T*(1+a)) ]
               / [ pi*t/T * (1 - (4a*t/T)^2) ]

    Removable singularities
        t = 0        ->  h = 1 + a*(4/pi - 1)
        t = +-T/(4a) ->  h = (a/sqrt2)*[(1+2/pi)sin(pi/4a) + (1-2/pi)cos(pi/4a)]
    """
    t = np.asarray(t, dtype=float)
    x = t / T
    h = np.empty_like(x)

    sing0 = np.abs(x) < 1e-12
    if alpha > 0:
        sing1 = np.abs(np.abs(x) - 1.0 / (4 * alpha)) < 1e-9
    else:
        sing1 = np.zeros_like(x, dtype=bool)
    good = ~(sing0 | sing1)

    xg = x[good]
    num = np.sin(np.pi * xg * (1 - alpha)) + \
          4 * alpha * xg * np.cos(np.pi * xg * (1 + alpha))
    den = np.pi * xg * (1 - (4 * alpha * xg) ** 2)
    h[good] = num / den

    h[sing0] = 1.0 + alpha * (4.0 / np.pi - 1.0)
    if alpha > 0 and sing1.any():
        h[sing1] = (alpha / np.sqrt(2.0)) * (
            (1 + 2 / np.pi) * np.sin(np.pi / (4 * alpha)) +
            (1 - 2 / np.pi) * np.cos(np.pi / (4 * alpha))
        )
    return h


# ----------------------------------------------------------------------


## 3. Task 23 — upsampled symbol impulse train


In [ ]:
def impulse_train(symbols, sps=sps):
    """Insert sps-1 zeros between symbols (zero-order upsampling)."""
    up = np.zeros(len(symbols) * sps)
    up[::sps] = symbols
    return up


# ----------------------------------------------------------------------


## 4. Spectrum and bandwidth helpers


In [ ]:
def spectrum(h, fs=fs, nfft=8192):
    H = np.fft.fftshift(np.fft.fft(h, nfft))
    f = np.fft.fftshift(np.fft.fftfreq(nfft, d=1 / fs))
    Hn = np.abs(H) / np.max(np.abs(H))
    return f, Hn


def bandwidth_metrics(h, fs=fs, nfft=8192):
    """Return (-3 dB one-sided BW, 99 % energy-containment BW)."""
    f, Hn = spectrum(h, fs, nfft)
    pos = f >= 0
    fp, Hp = f[pos], Hn[pos]

    # -3 dB point (first crossing of 1/sqrt(2))
    idx = np.where(Hp <= 1 / np.sqrt(2))[0]
    b3 = fp[idx[0]] if len(idx) else fp[-1]

    # 99 % energy containment
    P = Hp ** 2
    c = np.cumsum(P) / np.sum(P)
    b99 = fp[np.searchsorted(c, 0.99)]
    return b3, b99


# ----------------------------------------------------------------------


## 5. Build the filter bank


In [ ]:
t = np.arange(-SPAN / 2 * T, SPAN / 2 * T + Ts / 2, Ts)

rc  = {a: raised_cosine(t, a) for a in ALPHAS}
rrc = {a: root_raised_cosine(t, a) for a in ALPHAS}

# ======================================================================

print("h_RC(0)  =", [round(float(rc[a][len(t)//2]), 6)  for a in ALPHAS])
print("h_RRC(0) =", [round(float(rrc[a][len(t)//2]), 6) for a in ALPHAS],
      " <- equals 1 + a(4/pi - 1), NOT 1: RRC is not Nyquist on its own")


## Figure 1 — Pulse responses (Task 25)

RC pulses cross zero at every non-zero multiple of T. RRC pulses do **not** — that is the point.


In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))

ax[0, 0].plot(t, rect_pulse(t), lw=2, color="#b2182b")
ax[0, 0].set_title("Rectangular (NRZ) pulse")
ax[0, 0].set_ylim(-0.3, 1.25)

ax[0, 1].plot(t, sinc_pulse(t), lw=1.6, color="#2166ac")
ax[0, 1].set_title(r"Ideal sinc pulse  ($\alpha=0$)")

for a in ALPHAS:
    ax[1, 0].plot(t, rc[a], lw=1.5, label=rf"$\alpha$={a}")
    ax[1, 1].plot(t, rrc[a], lw=1.5, label=rf"$\alpha$={a}")
ax[1, 0].set_title("Raised cosine")
ax[1, 1].set_title("Root raised cosine")
ax[1, 0].legend(fontsize=8); ax[1, 1].legend(fontsize=8)

for a_ in ax.ravel():
    a_.axhline(0, color="k", lw=0.6)
    a_.set_xlabel("t / T"); a_.set_ylabel("h(t)")
    a_.grid(alpha=0.3)
    for k in range(-5, 6):
        a_.axvline(k * T, color="grey", lw=0.4, ls=":")

fig.suptitle("Figure 1 — Pulse responses", fontsize=13, fontweight="bold")
fig.tight_layout()
if SAVE: fig.savefig(f"{OUT}/fig1_pulse_responses.png", dpi=140)
plt.show()

# ======================================================================


## Figure 2 — Frequency responses (Task 25)

RC is strictly band-limited to (1+α)/2T and passes through −6 dB at f = 1/2T for every α.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))

f, Hrect = spectrum(rect_pulse(t))
ax[0].plot(f, 20 * np.log10(Hrect + 1e-12), color="k", ls="--",
           lw=1.1, label="Rectangular")
for a in ALPHAS:
    f, H = spectrum(rc[a])
    ax[0].plot(f, 20 * np.log10(H + 1e-12), lw=1.3, label=rf"RC $\alpha$={a}")
    f, H = spectrum(rrc[a])
    ax[1].plot(f, 20 * np.log10(H + 1e-12), lw=1.3, label=rf"RRC $\alpha$={a}")

for a_, ttl in zip(ax, ["Raised cosine vs rectangular", "Root raised cosine"]):
    a_.set_xlim(-2 / T, 2 / T); a_.set_ylim(-80, 5)
    a_.axvline(0.5 / T, color="k", ls="--", lw=0.8)
    a_.axvline(-0.5 / T, color="k", ls="--", lw=0.8)
    a_.set_xlabel("Frequency (×1/T)"); a_.set_ylabel("|H(f)| (dB)")
    a_.set_title(ttl); a_.grid(alpha=0.3); a_.legend(fontsize=8)
ax[0].text(0.53 / T, -75, "Nyquist\nedge 1/2T", fontsize=7)

fig.suptitle("Figure 2 — Frequency responses", fontsize=13, fontweight="bold")
fig.tight_layout()
if SAVE: fig.savefig(f"{OUT}/fig2_frequency_responses.png", dpi=140)
plt.show()

# ======================================================================


## Figure 3 — Pulse-shaped stream (Tasks 23, 26, 27)

Panel 3 is the Tx-RRC → Rx-RRC cascade: it lands back on the ±1 markers, restoring the Nyquist property.


In [ ]:
NSYM = 20
bits = rng.integers(0, 2, NSYM)
syms = 2.0 * bits - 1.0                       # BPSK  +/-1
train = impulse_train(syms)
t_tr = np.arange(len(train)) * Ts

fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

ax[0].stem(t_tr[::sps], train[::sps], basefmt=" ", linefmt="C0-", markerfmt="C0o")
ax[0].set_title("Task 23 — upsampled symbol impulse train (BPSK, 16 sps)")
ax[0].set_ylabel("amplitude")

for a in [0.25, 1.0]:
    y = np.convolve(train, rc[a], mode="full")
    d = (len(rc[a]) - 1) // 2
    y = y[d:d + len(train)]
    ax[1].plot(t_tr, y, lw=1.4, label=rf"RC $\alpha$={a}")
ax[1].stem(t_tr[::sps], syms, basefmt=" ", linefmt="k:", markerfmt="ko")
ax[1].set_title("Task 26 — RC-shaped stream; markers = ideal symbol decisions")
ax[1].set_ylabel("amplitude"); ax[1].legend(fontsize=8)

a = 0.25
y_tx = np.convolve(train, rrc[a], mode="full")
y_rx = np.convolve(y_tx, rrc[a], mode="full")
d = len(rrc[a]) - 1
y_rx = y_rx[d:d + len(train)]
y_rx /= np.max(np.abs(np.convolve(rrc[a], rrc[a])))   # unit peak per symbol
ax[2].plot(t_tr, y_rx, lw=1.4, color="#762a83",
           label=r"RRC$\otimes$RRC matched-filter output")
ax[2].stem(t_tr[::sps], syms, basefmt=" ", linefmt="k:", markerfmt="ko")
ax[2].set_title(r"Task 27 — cascaded Tx/Rx RRC ($\alpha$=0.25) sampled at kT")
ax[2].set_xlabel("time (×T)"); ax[2].set_ylabel("amplitude")
ax[2].legend(fontsize=8)

for a_ in ax:
    a_.axhline(0, color="k", lw=0.6); a_.grid(alpha=0.3)
    for k in range(NSYM):
        a_.axvline(k * T, color="grey", lw=0.35, ls=":")

fig.suptitle("Figure 3 — Pulse-shaped stream", fontsize=13, fontweight="bold")
fig.tight_layout()
if SAVE: fig.savefig(f"{OUT}/fig3_pulse_shaped_stream.png", dpi=140)
plt.show()

# ======================================================================


## Figure 4 — Bandwidth versus roll-off (Task 24)

Null-to-null bandwidth follows (1+α)/2T exactly; excess bandwidth rises linearly to 100 % at α = 1.


In [ ]:
alpha_sweep = np.linspace(0, 1, 21)
b3_l, b99_l, theo = [], [], []
for a in alpha_sweep:
    h = raised_cosine(t, a)
    b3, b99 = bandwidth_metrics(h)
    b3_l.append(b3); b99_l.append(b99)
    theo.append((1 + a) / (2 * T))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
ax[0].plot(alpha_sweep, theo, "k--", lw=2, label=r"theory $(1+\alpha)/2T$")
ax[0].plot(alpha_sweep, b99_l, "o-", ms=4, label="measured 99 % energy BW")
ax[0].plot(alpha_sweep, b3_l, "s-", ms=4, label="measured −3 dB BW")
ax[0].set_xlabel(r"roll-off $\alpha$"); ax[0].set_ylabel("bandwidth (×1/T)")
ax[0].set_title("Absolute bandwidth vs roll-off")
ax[0].grid(alpha=0.3); ax[0].legend(fontsize=8)

excess = 100 * (np.array(theo) - 0.5 / T) / (0.5 / T)
ax[1].plot(alpha_sweep, excess, "o-", ms=4, color="#1b7837")
ax[1].set_xlabel(r"roll-off $\alpha$")
ax[1].set_ylabel("excess bandwidth (%)")
ax[1].set_title("Excess bandwidth over the Nyquist minimum")
ax[1].grid(alpha=0.3)

fig.suptitle("Figure 4 — Bandwidth versus roll-off", fontsize=13, fontweight="bold")
fig.tight_layout()
if SAVE: fig.savefig(f"{OUT}/fig4_bandwidth_vs_rolloff.png", dpi=140)
plt.show()

# ======================================================================


## 9. Figure 5 — Eye diagrams, four-way comparison

400 random symbols overlaid one symbol period at a time. The red line is the ideal
sampling instant.


In [ ]:
NSYM_EYE = 400
b_e = rng.integers(0, 2, NSYM_EYE)
s_e = 2.0 * b_e - 1.0
tr_e = impulse_train(s_e)

def shaped(h):
    y = np.convolve(tr_e, h, mode="full")
    d = (len(h) - 1) // 2
    return y[d:d + len(tr_e)]

cases = [("Rectangular", rect_pulse(t)),
         (r"Sinc ($\alpha$=0)", sinc_pulse(t)),
         (r"RC $\alpha$=0.25", rc[0.25]),
         (r"RC $\alpha$=1.0", rc[1.0])]

fig, ax = plt.subplots(1, 4, figsize=(14, 3.8), sharey=True)
for a_, (nm, h) in zip(ax, cases):
    y = shaped(h)
    for k in range(4, NSYM_EYE - 4):
        seg = y[k * sps - sps: k * sps + sps + 1]
        a_.plot(np.arange(-sps, sps + 1) / sps, seg, color="#2166ac", lw=0.35, alpha=0.25)
    a_.axvline(0, color="r", ls="--", lw=0.9)
    a_.set_title(nm, fontsize=10); a_.set_xlabel(r"$t/T$"); a_.grid(alpha=0.3)
ax[0].set_ylabel("amplitude")
fig.suptitle("Figure 5 — Eye diagrams (red line = ideal sampling instant)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
if SAVE: fig.savefig(f"{OUT}/fig5_eye_diagrams.png", dpi=140)
plt.show()

# ======================================================================


## 10. Figure 6 — The bandwidth–timing trade-off

Worst-case eye opening = |h(Δt)| − Σ|h(kT+Δt)| over k ≠ 0, i.e. the ISI margin under the
worst possible data pattern. **This is the cell that actually demonstrates the trade-off**:
at Δt = 0 every roll-off scores exactly 1.0, so the Nyquist criterion alone cannot tell
them apart.


In [ ]:
offs = np.linspace(-0.5, 0.5, 101)
eye_open = {}
for a in ALPHAS:
    op = []
    ks_ = np.arange(-5, 6)
    for dt in offs:
        # evaluate the closed-form pulse directly at t = kT + dt (no indexing)
        v = raised_cosine((ks_ + dt) * T, a)
        wanted = v[5]                              # desired sample at k=0
        isi = np.sum(np.abs(np.delete(v, 5)))      # worst-case ISI
        op.append(abs(wanted) - isi)
    eye_open[a] = np.array(op)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for a in ALPHAS:
    ax[0].plot(offs, eye_open[a], lw=1.6, label=rf"$\alpha$={a}")
ax[0].axhline(0, color="k", lw=0.8, ls=":")
ax[0].set_xlabel(r"timing offset $\Delta t/T$")
ax[0].set_ylabel("worst-case eye opening")
ax[0].set_title("Eye opening vs timing error"); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=8)

jit = []
for a in ALPHAS:
    closed = np.where(eye_open[a] <= 0)[0]
    if len(closed):
        jit.append(min(abs(offs[closed]).min(), 0.5))
    else:
        jit.append(0.5)
ax[1].bar([str(a) for a in ALPHAS], jit, color="#1b7837", width=0.55)
ax[1].set_xlabel(r"roll-off $\alpha$"); ax[1].set_ylabel(r"$|\Delta t|/T$ at eye closure")
ax[1].set_title("Timing margin before the eye closes"); ax[1].grid(alpha=0.3, axis="y")

fig.suptitle("Figure 6 — Bandwidth–timing trade-off", fontsize=13, fontweight="bold")
fig.tight_layout()
if SAVE: fig.savefig(f"{OUT}/fig6_timing_sensitivity.png", dpi=140)
plt.show()


## 11. Mandatory validation

Validation 1 is the one named in the assignment sheet: *sample the overall RC response at
integer symbol intervals and tabulate the values.* Validations 2–8 support it.


In [ ]:
lines = []
lines.append("MANDATORY VALIDATION 1 — raised cosine h(kT), k = -5..5")
lines.append("=" * 74)
hdr = f"{'k':>4} " + "".join(f"{'a='+str(a):>16}" for a in ALPHAS)
lines.append(hdr); lines.append("-" * 74)
ks = np.arange(-5, 6)
tab_rc = {}
for a in ALPHAS:
    tab_rc[a] = raised_cosine(ks * T, a)
for i, k in enumerate(ks):
    lines.append(f"{k:>4} " + "".join(f"{tab_rc[a][i]:>16.3e}" for a in ALPHAS))
lines.append("-" * 74)
worst = max(np.max(np.abs(np.delete(tab_rc[a], 5))) for a in ALPHAS)
lines.append(f"Peak |h(kT)| for k != 0 across all alpha : {worst:.3e}")
lines.append(f"h(0) for every alpha                     : "
             f"{[round(float(tab_rc[a][5]), 12) for a in ALPHAS]}")
lines.append("")

lines.append("MANDATORY VALIDATION 2 — cascaded RRC(Tx)*RRC(Rx) at k T")
lines.append("=" * 74)
lines.append(f"{'k':>4} " + "".join(f"{'a='+str(a):>16}" for a in ALPHAS))
lines.append("-" * 74)
casc = {}
for a in ALPHAS:
    c = np.convolve(rrc[a], rrc[a])
    c = c / np.max(np.abs(c))                 # normalise peak to 1
    centre = (len(c) - 1) // 2
    casc[a] = np.array([c[centre + k * sps] for k in ks])
for i, k in enumerate(ks):
    lines.append(f"{k:>4} " + "".join(f"{casc[a][i]:>16.3e}" for a in ALPHAS))
lines.append("-" * 74)
for a in ALPHAS:
    err = np.max(np.abs(casc[a] - tab_rc[a]))
    lines.append(f"alpha={a:<5} max |RRC*RRC - RC| at symbol instants = {err:.3e}")
lines.append("")

lines.append("MANDATORY VALIDATION 3 — ISI budget of the shaped waveform")
lines.append("=" * 74)
lines.append(f"{'alpha':>7}{'peak ISI':>14}{'sum|ISI|':>14}"
             f"{'eye opening':>14}{'PAPR (dB)':>12}")
lines.append("-" * 74)
for a in ALPHAS:
    tails = np.delete(tab_rc[a], 5)
    y = np.convolve(train, rc[a], mode="full")
    d = (len(rc[a]) - 1) // 2
    y = y[d:d + len(train)]
    papr = 20 * np.log10(np.max(np.abs(y)) / np.sqrt(np.mean(y ** 2)))
    lines.append(f"{a:>7}{np.max(np.abs(tails)):>14.3e}"
                 f"{np.sum(np.abs(tails)):>14.3e}"
                 f"{1 - np.sum(np.abs(tails)):>14.6f}{papr:>12.2f}")
lines.append("")

lines.append("MANDATORY VALIDATION 4 — bandwidth table (RC)")
lines.append("=" * 74)
lines.append(f"{'alpha':>7}{'theory (1+a)/2T':>20}{'99% BW':>12}"
             f"{'-3 dB BW':>12}{'excess %':>12}")
lines.append("-" * 74)
for a in ALPHAS:
    b3, b99 = bandwidth_metrics(rc[a])
    th = (1 + a) / (2 * T)
    lines.append(f"{a:>7}{th:>20.4f}{b99:>12.4f}{b3:>12.4f}"
                 f"{100*(th-0.5)/0.5:>12.1f}")
lines.append("")

lines.append("MANDATORY VALIDATION 5 — singularity limits vs numerical neighbour")
lines.append("=" * 74)
eps = 1e-7
for a in [0.25, 0.5, 1.0]:
    ts = T / (2 * a)
    lines.append(f"RC  alpha={a}: analytic h(T/2a)={raised_cosine(np.array([ts]), a)[0]:.10f}"
                 f"   numeric h(T/2a+1e-7)={raised_cosine(np.array([ts+eps]), a)[0]:.10f}")
for a in [0.25, 0.5, 1.0]:
    ts = T / (4 * a)
    lines.append(f"RRC alpha={a}: analytic h(T/4a)={root_raised_cosine(np.array([ts]), a)[0]:.10f}"
                 f"   numeric h(T/4a+1e-7)={root_raised_cosine(np.array([ts+eps]), a)[0]:.10f}")

lines.append("")
lines.append("MANDATORY VALIDATION 6 — diagnostic: is the cascade error truncation?")
lines.append("=" * 74)
lines.append(f"{'span (symbols)':>16}" + "".join(f"{'a='+str(a):>16}" for a in ALPHAS))
lines.append("-" * 74)
for SP in [10, 20, 40, 80]:
    tt = np.arange(-SP / 2 * T, SP / 2 * T + Ts / 2, Ts)
    row = f"{SP:>16}"
    for a in ALPHAS:
        g = root_raised_cosine(tt, a)
        c = np.convolve(g, g); c /= np.max(np.abs(c))
        ctr = (len(c) - 1) // 2
        v = np.array([c[ctr + k * sps] for k in ks])
        row += f"{np.max(np.abs(v - raised_cosine(ks * T, a))):>16.3e}"
    lines.append(row)
lines.append("-" * 74)
lines.append("Error falls monotonically with span => residual is window truncation,")
lines.append("not an error in the closed-form filter definitions.")

report = "\n".join(lines)
print(report)
if SAVE:
    with open(f"{OUT}/validation_tables.txt", "w") as f:
        f.write(report + "\n")

# ======================================================================


## 12. Validations 7 and 8


In [ ]:
extra = []
extra.append("")
extra.append("VALIDATION 7 — bandwidth vs timing trade-off (the real trade table)")
extra.append("=" * 74)
extra.append(f"{'alpha':>7}{'BW (1+a)/2T':>15}{'eye @ dt=0':>13}"
             f"{'eye @ 5%T':>12}{'eye @ 10%T':>13}{'closes at':>12}")
extra.append("-" * 74)
for a, j in zip(ALPHAS, jit):
    e0 = eye_open[a][np.argmin(np.abs(offs - 0.0))]
    e5 = eye_open[a][np.argmin(np.abs(offs - 0.05))]
    e10 = eye_open[a][np.argmin(np.abs(offs - 0.10))]
    cl = f"{j:.3f}T" if j < 0.5 else ">0.5T"
    extra.append(f"{a:>7}{(1+a)/2:>15.3f}{e0:>13.4f}{e5:>12.4f}{e10:>13.4f}{cl:>12}")
extra.append("-" * 74)
extra.append("Eye opening = |h(0+dt)| - sum|h(kT+dt)|, k!=0  (worst-case ISI over all")
extra.append("data patterns). At dt=0 every RC gives 1.0 -> that metric alone cannot")
extra.append("discriminate; the trade-off only becomes visible under timing error.")
extra.append("")
extra.append("VALIDATION 8 — four-way pulse comparison")
extra.append("=" * 74)
extra.append(f"{'pulse':>16}{'h(kT) k!=0':>14}{'tail decay':>13}"
             f"{'null-null BW':>14}{'1st sidelobe':>14}")
extra.append("-" * 74)
for nm, h, decay, bw in [("Rectangular", rect_pulse(t), "1/f in freq", "infinite"),
                          ("Sinc a=0", sinc_pulse(t), "1/t", "0.500/T"),
                          ("RC a=0.25", rc[0.25], "1/t^3", "0.625/T"),
                          ("RC a=1.0", rc[1.0], "1/t^3", "1.000/T"),
                          ("RRC a=0.25", rrc[0.25], "1/t^3", "0.625/T")]:
    ctr = (len(h) - 1) // 2
    v = np.array([h[ctr + k * sps] for k in np.arange(-5, 6)])
    worst_ = np.max(np.abs(np.delete(v, 5)))
    f_, H_ = spectrum(h)
    pos = f_ > 0.05
    sl = 20 * np.log10(np.max(H_[pos & (f_ > (1.2 if nm == "Rectangular" else 1.05))]) + 1e-12)
    extra.append(f"{nm:>16}{worst_:>14.3e}{decay:>13}{bw:>14}{sl:>13.1f}dB")
extra.append("-" * 74)
extra.append("Only the RC rows show machine-zero at k!=0. RRC does not -> not Nyquist alone.")

if SAVE:
    with open(f"{OUT}/validation_tables.txt", "a") as fh:
        fh.write("\n".join(extra) + "\n")
print("\n".join(extra))


## 13. Optional — save everything and download (Colab)

Set `SAVE = True` in cell 1, re-run all cells, then run this to pull a ZIP to your laptop.


In [ ]:
# Colab only — bundles every saved figure and table into one download
import os, shutil
try:
    from google.colab import files
    shutil.make_archive("experiment7_outputs", "zip", OUT)
    files.download("experiment7_outputs.zip")
except ImportError:
    print("Not running in Colab — files are in:", os.path.abspath(OUT))
